# Gait-ViViT: A Video Processing Model for Parkinson's Disease Detection

In [ ]:
# Required libraries.
import os
import json
import numpy as np
import cv2
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as v2
from torchvision import tv_tensors
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2- Frame Extraction and Preprocessing

After the synthetic videos have been created and saved, the next step consists of extracting and cleaning the frames that will be used to train the model.

Since the synthetic videos are created based on files coming from different datasets, they might have different durations and the skeletons could be visible at different intervals.

For this reason, a **frame extraction interval** is defined depending on the video source and on the action performed by the subject.

- Since the videos in the Kaggle MMU Dataset are shorter and more consistent in the actions performed, the frames will be extracted **throughout the entire video**.
- Since the videos in the internal dataset have different lengths and show different actions, the following intervals are used:
  - For videos representing the `stairs_down` action, the frames will be extracted from **half of the video duration up to the end of the video**.
  - For videos representing the `stairs_up` action, the frames will be extracted **from the start of the video up to $75\%$ of the video duration**.
  - For videos representing the `walk` action, the frames will be extracted **from half of the video duration up to $70\%$ of the video duration**.

Additionally, these synthetic videos still present some **noise**, such as the presence of duplicated skeletons, requiring to **clean the videos and the extracted frames** in order to avoid training errors.

The frame extraction pipeline is implemented in the `frame_extraction` function, which **reads the input video and the corresponding keypoints file** and uses the **frame keys** contained in the keypoints file to **extract the video frames**.

Next, the function determines **which frames belong to the chosen frame extraction interval** and it **filters out all invalid frames** using the `frame_confidence` function, which reads the information associated to a frame in order to determine whether to keep it or discard it.

In particular, the function **discards** a frame under one of the following conditions:
- Keypoint information about the frame is **not found** in the file.
- **No skeleton** is detected in the frame.
- **Multiple skeletons** are detected in the frame.
- The **average confidence score** of the keypoints is **below the chosen threshold**.

After filtering out the invalid frames, the main function uses the `np.linspace()` function to **uniformly sample `num_frames` valid frames** from the extraction interval, eventually duplicating frames if less than `num_frames` frames are found.

Lastly, each sampled frame is processed by **converting the colour scheme from BGR to RGB, applying a median filter and resizing it to the target size**, eventually duplicating other frames or using a black placeholder frame whenever the extraction fails.

At the end of the pipeline, the function will return an array of shape $(num\_frames, 224, 224, 3)$.

In [ ]:
def frame_confidence(keypoints_input, frame_key, threshold=0.5):
  # This function checks whether the average keypoint confidence is above the chosen threshold.
  if isinstance(keypoints_input, str):
    # If the file path is given, open the corresponding file first.
    with open(keypoints_input, "r") as f:
      keypoints_data = json.load(f)
  else:
    keypoints_data = keypoints_input
  frame_info = None

  # JSON keypoints from the Kaggle Dataset are stored as lists.
  if isinstance(keypoints_data, list):
    clean_target = str(frame_key).split(".")[0] # Extract the frame ID regardless of whether the key is passed as x, "x" or "x.jpg".
    for item in keypoints_data:
      img_id = str(item.get("image_id", ""))
      if img_id == str(frame_key) or img_id.split(".")[0] == clean_target:
        frame_info = item
        break

  # JSON keypoints from the Internal Dataset are stored as nested dictionaries.
  elif isinstance(keypoints_data, dict):
    clean_target = str(frame_key).split(".")[0] # Extract the frame ID regardless of whether the key is passed as x, "x" or "x.jpg".
    possible_keys = [frame_key, str(frame_key), clean_target, f"{clean_target}.jpg"]
    for k in possible_keys:
      if k in keypoints_data:
        frame_info = keypoints_data[k]
        break

  if frame_info is None:
    # Fallback if the frame is not found.
    # print(f"Frame {frame_key} not found.") # Uncomment for debugging.
    return False

  # Extract information from the frame.
  bodies = []
  if isinstance(frame_info, dict):
    bodies = frame_info.get("bodies") or frame_info.get("people") or [frame_info]
  elif isinstance(frame_info, list):
    bodies = frame_info

  if not bodies:
    # Fallback if no body is detected.
    # print(f"No body was found in frame {frame_key}.") # Uncomment for debugging.
    return False
  elif len(bodies) != 1:
    # Fallback if more skeletons are detected in the frame.
    # print(f"Multiple skeletons detected in frame {frame_key}.") # Uncomment for debugging.
    return False

  # Compute the average confidence score and the number of visible keypoints for the frame.
  avg_confidence = -1.0 # Default value.
  body = bodies[0]

  keypoints = body.get("joints") or body.get("keypoints", [])
  if not keypoints:
    # Fallback in case no keypoints are found.
    return False

  scores = np.array(keypoints[2::3]) # Take just the confidence scores.
  if scores.size > 0 and scores.any():
    avg_confidence = max(avg_confidence, float(np.mean(scores)))

  return (avg_confidence > threshold)

In [ ]:
def frame_extraction_pipeline(video_path, keypoints_path, interval=(0.0, 1.0), target_size=(224, 224), threshold=0.5, num_frames=32):
  # Read the video and the keypoints.
  cap = cv2.VideoCapture(video_path)
  if not cap.isOpened():
    raise ValueError(f"Error when opening {video_path}.")

  total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

  with open(keypoints_path, "r") as f:
    keypoints_data = json.load(f)

  # Extract all the frame keys in the JSON file and convert them into sorted values.
  if isinstance(keypoints_data, list):
    # JSON keypoints from the Kaggle Dataset are stored as lists.
    raw_keys = [item.get("image_id") for item in keypoints_data if isinstance(item, dict) and "image_id" in item]
  elif isinstance(keypoints_data, dict):
    # JSON keypoints from the Internal Dataset are stored as nested dictionaries.
    raw_keys = list(keypoints_data.keys())
  else:
    raw_keys = []

  valid_frames = []
  for k in raw_keys:
    try:
      n = int(str(k).split(".")[0])
      valid_frames.append(n)
    except(ValueError, TypeError):
      # Fallback in case the key extraction fails.
      continue

  valid_frames = sorted(list(set(valid_frames))) # Eliminate duplicates and sort the frame keys.

  # Define the extraction interval.
  start_position = int(len(valid_frames) * interval[0])
  end_position = max(start_position, int(len(valid_frames) * interval[1]) - 1)
  interval_frames = valid_frames[start_position:end_position+1]

  # Filter out invalid or low-confidence frames, as well as frames presenting more than one skeleton.
  clean_valid_frames = [idx for idx in interval_frames if frame_confidence(keypoints_data, idx, threshold)]

  # Use np.linspace() to see which frames should be sampled.
  if clean_valid_frames:
    # Some valid frames have been found, so sample the positions and retrieve the corresponding frames.
    positions = np.linspace(0, len(clean_valid_frames) - 1, num=num_frames, dtype=int)
    frame_indices = [clean_valid_frames[p] for p in positions]
  else:
    # Fallback in case no valid frame has been found.
    print(f"No valid frame found for {video_path}.")
    frame_indices = []

  # Extract and clean the chosen frames.
  frames = []
  adjacent_frame = None # Buffer for duplicating corrupted/invalid frames.

  if frame_indices:
    # Iterate through the valid frames.
    for idx in frame_indices:
      cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
      ret, frame = cap.read()
      if ret and frame is not None:
        # The frame is valid and gets processed.
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) # Convert from OpenCV's BGR scheme to the RGB scheme.
        if "internal_videos" in video_path:
          # The frame_confidence() function does not eliminate all noise in these videos due to AlphaPose bugs in the rendering.
          rgb_frame = rgb_frame[:, int(0.4 * rgb_frame.shape[1]):, :]
        smoothed_frame = cv2.medianBlur(rgb_frame, 3) # Use OpenCV's native median filter.
        resized_frame = cv2.resize(smoothed_frame, target_size, interpolation=cv2.INTER_AREA)
        frames.append(resized_frame)
        adjacent_frame = resized_frame # Save the latest valid frame as a buffer.
      else:
        # The frame is not valid and gets replaced by the latest valid frame.
        frames.append(adjacent_frame)

    # Since invalid/corrupted frames might also appear at the start of the sequence, duplicate the first valid frames for those slots as well.
    to_duplicate = next((f for f in frames if f is not None), None) # Extract the first valid frame or get None if no valid frame exists.
    if to_duplicate is None:
      # Fallback in case no valid frame has been found.
      for i in range(num_frames):
        frames[i] = np.zeros((*target_size, 3), dtype=np.uint8)
    else:
      # Some valid frame to duplicate is found.
      for i in range(num_frames):
        if frames[i] is None:
          frames[i] = to_duplicate

  else:
    # Fallback in case no valid frame has been found.
    for _ in range(num_frames):
      zero_frame = np.zeros((*target_size, 3), dtype=np.uint8)
      frames.append(zero_frame)

  cap.release()

  frames_array = np.array(frames)
  print(f"{len(frames_array)} frames found for {video_path}. Shape: {frames_array.shape}")
  return frames_array

## Frame Storage

To avoid repeating the extraction procedure everytime, each array is **converted into a tensor** that **stored into a new dataframe containing a path to said tensor and the label associated to the original synthetic video**.

This step is useful as it will allow to load the ready tensors using just a custom `Dataset` class and the corresponding `DataLoader`.

In [ ]:
video_df = pd.read_csv("/content/drive/MyDrive/bachelor_thesis/dataframes/unified_dataset.csv")

for _, row in video_df.iterrows():
  # Extract the video path and recover the corresponding keypoints.
  video_path = row["path"]
  json_path = (video_path.replace("videos", "json")).replace(".mp4", ".json")

  # Create the tensor directories.
  tensor_path = (video_path.replace("videos", "tensors")).replace(".mp4", ".pt")
  os.makedirs(os.path.dirname(tensor_path), exist_ok=True)

  # Since the procedure may interrupt, skip any video whose frames have been previously extracted.
  if os.path.exists(tensor_path):
    continue

  # Frame extraction.
  print(f"Extracting frames from {video_path}.")
  if row["source"] == "Kaggle":
    # Kaggle Video frame extraction: Extract frames throughout the entire video and with a lower confidence threshold.
    frames = frame_extraction_pipeline(video_path, json_path, (0.0, 1.0), (224, 224), 0.15, 32)
  elif row["source"] == "Internal":
    # Internal Video frame extraction: Extract frames at different intervals and with a higher confidence threshold.
    if "stairs_down" in video_path:
      # Stairs down action.
      frames = frame_extraction_pipeline(video_path, json_path, (0.5, 1.0), (224, 224), 0.5, 32)
    elif "stairs_up" in video_path:
      frames = frame_extraction_pipeline(video_path, json_path, (0.0, 0.75), (224, 224), 0.5, 32)
    elif "walk" in video_path:
      frames = frame_extraction_pipeline(video_path, json_path, (0.5, 0.7), (224, 224), 0.5, 32)
    else:
      # Fallback for unknown actions.
      frames = None # Fallback placeholder value.
      print(f"Skipping unknown video {video_path}.")
  else:
    # Fallback for unknown videos.
    frames = None # Fallback placeholder value.
    print(f"Unknown source {row["source"]}: skipping {video_path}.")

  if frames is not None:
    # Convert the found frames from NumPy to PyTorch.
    print(f"Saving to {tensor_path}.")
    frame_tensor = torch.from_numpy(frames)
    torch.save(frame_tensor, tensor_path)

print("Frame extraction and storage successfully completed.")

In [ ]:
video_df = pd.read_csv("/content/drive/MyDrive/bachelor_thesis/dataframes/unified_dataset.csv")
tensors = []

for _, row in video_df.iterrows():
  # Extract the video path and recover the corresponding frame tensor.
  video_path = row["path"]
  tensor_path = (video_path.replace("videos", "tensors")).replace(".mp4", ".pt")

  # Skip videos for which no valid frame was found.
  tensor = torch.load(tensor_path)
  if torch.all(tensor == 0).item():
    print(f"No valid frames found for {video_path}: Skipping.")
    continue

  # Add the original video's ID and label, as well as the frame tensor, to the dataframe.
  tensors.append({
      "video_id": row["video_id"],
      "source": row["source"],
      "path": tensor_path,
      "parkinson": row["parkinson"]
  })

# Tensor dataframe aggregation.
tensor_df = pd.DataFrame(tensors)

print(f"Total tensors: {len(tensor_df)}.")

print("Count based on the source dataset:")
print(tensor_df['source'].value_counts())
print("\nClass balance (0 = Healthy, 1 = Parkinson):")
print(tensor_df['parkinson'].value_counts())

df_dir = "/content/drive/MyDrive/bachelor_thesis/dataframes"
os.makedirs(df_dir, exist_ok=True)
tensor_df.to_csv("/content/drive/MyDrive/bachelor_thesis/dataframes/tensor_dataset.csv", index=False)

print("Tensor dataframe successfully created.")

In [ ]:
class GaitViViTDataset(Dataset):
  def __init__(self, tensor_df, frames_per_video=32, transform=None):
    # Load the dataframe for initialization.
    if isinstance(tensor_df, str):
      self.data = pd.read_csv(tensor_df)
    else:
      self.data = tensor_df.reset_index(drop=True)
    self.frames_per_video = frames_per_video
    self.transform = transform

  def __len__(self):
    # Find the number of elements in the dataframe.
    return len(self.data)

  def __getitem__(self, idx):
    # Load, and eventually transform, a specific tensor and recover the label associated to the original video.
    tensor_path = self.data.iloc[idx]["path"]
    tensor = torch.load(tensor_path)

    # Convert the tensor from uint8 to float32 and permute its shape from (F, H, W, C) to (C, F, H, W) for training compatibility.
    tensor = tensor.float()
    tensor = tensor.permute(3, 0, 1, 2).contiguous()
    tensor_shape = tensor.shape

    # Apply ImageNet normalization.
    if tensor.max() > 1:
      tensor = tensor / 255.0 # Scaling.
    mean = tensor.new_tensor([0.485, 0.456, 0.406]).view(3, 1, 1, 1) # Make mean broadcast-compatible.
    std = tensor.new_tensor([0.229, 0.224, 0.225]).view(3, 1, 1, 1) # Make std broadcast-compatible.
    tensor = (tensor - mean) / std # Normalization.

    if self.transform:
      # Apply any transformation to the loaded tensor.
      tensor = tv_tensors.Video(tensor) # Make sure the transformation is applied to ALL frames.
      tensor = self.transform(tensor)
    label = self.data.iloc[idx]["parkinson"]
    return tensor, label

## Frame Visualization

After extracting frames from a video, it is possible to load the resulting tensor in order to look at the frames that have been extracted.

In [ ]:
# Load a sample tensor and look at its shape.
test = torch.load("/content/drive/MyDrive/bachelor_thesis/internal_tensors/dataset_blurred/FirstRun/Alessio/rgb/stairs_down/Alessio_stairs_down_1.pt")
print(f"Shape of the tensor: {test.shape}")

# Check whether the tensor contains black or discarded frames.
is_all_zero = torch.all(test == 0).item()
print(f"Is the tensor entirely composed by zeros? {is_all_zero}")

has_values = torch.any((test != 0) & ~torch.isnan(test)).item()
print(f"Are there valid values? {has_values}")

nan_frames = torch.isnan(test[:, 0, 0, 0]).sum().item() # Check just the first pixel.
print(f"Discarded frames: {nan_frames}.")

In [ ]:
def frame_visualization(tensor_file):
  # Load the tensor.
  frames = torch.load(tensor_file)
  n_frames = frames.shape[0]

  # Set grid dimensions.
  cols = 8
  rows = (n_frames + cols - 1) // cols

  # Create the main image and the subplots.
  fig, axes = plt.subplots(rows, cols, figsize=(20, 2.5 * rows))
  axes = axes.flatten()

  for i in range(n_frames):
    ax = axes[i]
    frame = frames[i].numpy()

    if np.isnan(frame).any():
      # If the frame was discarded, show a black image with a text.
      blk = np.zeros((224, 224, 3), dtype=np.uint8)
      ax.imshow(blk)
      ax.text(112, 112, "NaN", color="red", ha="center", va="center", fontsize=12, weight="bold")
    else:
      ax.imshow(frame.astype(np.uint8))

    # Add a title to each subplot to indicate the frame.
    ax.set_title(f"Frame {i}")
    ax.axis("off")

  for j in range(n_frames, len(axes)):
    axes[j].axis("off")

  plt.tight_layout()
  plt.show()